In [67]:
from binance.um_futures import UMFutures
from database.adatabase import ADatabase
from binance_parameter_creator.binance_parameter_creator import BinanceParameterCreator as bpc
from binance_strategy.binance_strategy_factory import BinanceStrategyFactory
from crypto_parameter.acrypto_parameter import ACryptoParameter
from time import sleep
import pandas as pd
from datetime import datetime, timedelta
from binance.um_futures import UMFutures
from database.adatabase import ADatabase
from time import sleep
import pandas as pd
import random
import matplotlib.pyplot as plt
import pytz
kst_timezone = pytz.timezone('Asia/Seoul')

In [68]:
db = ADatabase("sapling")
db.cloud_connect()
bots = db.retrieve("crypto_bots")
keys = db.retrieve("crypto_secrets")
parameters = db.retrieve("crypto_parameter")
db.disconnect()
## parameters
for bot in bots.iterrows():
    user = bot[1]["username"]
    live = bot[1]["live"]
    if live == True:
        parameter = parameters[parameters["username"]==user].to_dict("records")[0]
        ticker = parameter["ticker"]
        secret = keys[keys["username"]==user]["bsecret"].item()
        key = keys[keys["username"]==user]["bkey"].item()
        umf = UMFutures(key,secret)
        account = umf.account()
        trades = umf.get_account_trades("XRPUSDT")
        trades = pd.DataFrame(trades)
        print(account["totalUnrealizedProfit"])
        positions = pd.DataFrame(account["positions"])
        xrp_positions = positions[positions["symbol"]==ticker]
        orders = pd.DataFrame(umf.get_all_orders("XRPUSDT"))
        new_orders = orders[orders["status"]=="NEW"]
        

-0.01228200


In [69]:
xrp_positions

,symbol,initialMargin,maintMargin,unrealizedProfit,positionInitialMargin,openOrderInitialMargin,leverage,isolated,entryPrice,breakEvenPrice,maxNotional,positionSide,positionAmt,notional,isolatedWallet,updateTime,bidNotional,askNotional
122,XRPUSDT,5.50118147,0.41258859,-0.01228200,5.50118147,0,15,True,0.5502,0.5504751,750000,BOTH,150.0,82.51771800,5.52125276,1708986678369,0,0


In [70]:
new_orders

,orderId,symbol,status,clientOrderId,price,avgPrice,origQty,executedQty,cumQuote,timeInForce,...,workingType,priceMatch,selfTradePreventionMode,goodTillDate,priceProtect,origType,time,updateTime,activatePrice,priceRate
457,50928178394,XRPUSDT,NEW,piySLb3k0bUBAkEs7xkcnv,0,0.00000,150,0,0,GTC,...,CONTRACT_PRICE,NONE,NONE,0,False,TRAILING_STOP_MARKET,1708986679520,1708986679520,0.5527,0.1


In [71]:
trades["date"] = [datetime.utcfromtimestamp(float(x) // 1000.0).replace(tzinfo=pytz.utc).astimezone(kst_timezone) for x in trades["time"]]

In [72]:
trades["realizedPnl"] = [float(x) for x in trades["realizedPnl"]]
trades["commission"] = [float(x) for x in trades["commission"]]
trades["price"] = [float(x) for x in trades["price"]]
trades["w/l"] = ["W" if x > 0 else "L" if x < 0 else "N" for x in trades["realizedPnl"]]
trades = trades[trades["date"]>datetime(2024,2,22).replace(tzinfo=pytz.utc).astimezone(kst_timezone)]
trades = trades.groupby(["date","w/l"]).agg({"price":"mean","commission":"sum","realizedPnl":"sum"}).reset_index()
trades["agg_pnl"] = trades["realizedPnl"].cumsum()
trades["agg_commission"] = trades["commission"].cumsum()
trades["pnl"] = trades["agg_pnl"] - trades["agg_commission"]
trades["entry_price"] = trades["price"].shift(1)
trades["price_diff"] = trades["price"].pct_change() * 100
# trades = trades[trades["w/l"]!="N"]
trades[["date","price","price_diff","w/l","realizedPnl","agg_pnl","commission","agg_commission","pnl"]].to_csv("trades.csv")